# 02 – Data Understanding: Diccionario de Datos

**Proyecto:** Predicción de subempleo por insuficiencia de horas – EPEN 2024 (INEI Perú)  
**Objetivo:** Revisar el significado, tipo, rango y calidad de las variables del dataset real.

In [3]:
import pandas as pd
import numpy as np
import os

# ─── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.getcwd()
PROJECT_DIR   = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
SNAPSHOT_PATH = os.path.join(PROJECT_DIR, 'data', 'raw', 'epen2024_raw_snapshot.csv')

print(f'Snapshot: {SNAPSHOT_PATH}')
print(f'Existe:   {os.path.exists(SNAPSHOT_PATH)}')

# ─── Carga ────────────────────────────────────────────────────────────────────
df = pd.read_csv(SNAPSHOT_PATH, encoding='utf-8-sig', low_memory=False)
print(f'\nFilas: {df.shape[0]:,} | Columnas: {df.shape[1]}')

Snapshot: c:\Users\ADMIN\Desktop\ML_PROYECTO_26_1\ml_project\data\raw\epen2024_raw_snapshot.csv
Existe:   True

Filas: 52,251 | Columnas: 132


## 1. Diccionario de Variables (selección relevante para el proyecto)

In [4]:
diccionario = pd.DataFrame([
    # --- Diseño muestral ---
    {'Variable': 'REGION',   'Tipo': 'Categórica', 'Descripción': 'Región geográfica',                               'Rango/Valores': '1=Lima Met. · 2=Resto urbano · 3=Rural',                          'Uso': 'Filtro/cobertura'},
    {'Variable': 'ESTRATO',  'Tipo': 'Categórica', 'Descripción': 'Estrato geográfico del diseño muestral',          'Rango/Valores': '1–8',                                                             'Uso': 'Diseño muestral'},
    {'Variable': 'RESIDENT', 'Tipo': 'Categórica', 'Descripción': 'Residente habitual del hogar',                    'Rango/Valores': '0=No · 1=Sí · 9=Omisión',                                        'Uso': 'Filtro'},
    # --- Demográficas ---
    {'Variable': 'C207',     'Tipo': 'Categórica', 'Descripción': 'Sexo',                                            'Rango/Valores': '1=Hombre · 2=Mujer',                                              'Uso': 'Predictor'},
    {'Variable': 'C208',     'Tipo': 'Numérica',   'Descripción': 'Edad en años cumplidos',                          'Rango/Valores': '0–98 (99=missing)',                                               'Uso': 'Predictor / filtro ≥14'},
    {'Variable': 'C203',     'Tipo': 'Categórica', 'Descripción': 'Relación de parentesco con el jefe del hogar',    'Rango/Valores': '1=Jefe … 11=Otro no pariente · 98=No residente',                  'Uso': 'Predictor'},
    # --- Condición de actividad ---
    {'Variable': 'OCUP300',  'Tipo': 'Categórica', 'Descripción': 'Nivel de ocupación / condición de actividad',     'Rango/Valores': '1=Ocupado · 2=Desoc. abierto · 3=Desoc. oculto · 4=Inactivo',   'Uso': 'Filtro (solo ocupados)'},
    # --- TARGET ---
    {'Variable': 'P209H',    'Tipo': 'Binaria',    'Descripción': 'Voluntad Y disponibilidad para trabajar más horas','Rango/Valores': '1=Sí · 2=No · vacío=No aplica',                                  'Uso': 'TARGET'},
    # --- Leakage ---
    {'Variable': 'C333',     'Tipo': 'Categórica', 'Descripción': '¿Quería trabajar más horas?',                     'Rango/Valores': '1=Sí · 2=No',                                                    'Uso': 'EXCLUIR (leakage)'},
    {'Variable': 'C334',     'Tipo': 'Categórica', 'Descripción': '¿Estuvo disponible para trabajar más horas?',     'Rango/Valores': '1=Sí · 2=No',                                                    'Uso': 'EXCLUIR (leakage)'},
    # --- Empleo y formalidad ---
    {'Variable': 'C310',     'Tipo': 'Categórica', 'Descripción': 'Categoría ocupacional',                           'Rango/Valores': '1=Empleador · 2=Independiente · 3=Empleado/obrero · 4–10=otros', 'Uso': 'Predictor'},
    {'Variable': 'C311',     'Tipo': 'Categórica', 'Descripción': 'Tipo de entidad donde trabaja',                   'Rango/Valores': '1=FF.AA. · 2=Adm.pública · 3=Emp.pública · 4=Service · 5=Privada · 6=Otra','Uso': 'Predictor'},
    {'Variable': 'C312',     'Tipo': 'Categórica', 'Descripción': 'Registro del negocio/empresa en SUNAT',           'Rango/Valores': '1=Persona jurídica · 2=Natural con RUC · 3=No registrado · 4=No sabe','Uso': 'Predictor'},
    {'Variable': 'C313',     'Tipo': 'Categórica', 'Descripción': 'Lleva libros contables (SUNAT)',                  'Rango/Valores': '1=Sí · 2=No · 3=No sabe',                                        'Uso': 'Predictor'},
    {'Variable': 'C317',     'Tipo': 'Categórica', 'Descripción': 'Tamaño del negocio/empresa (N° trabajadores)',    'Rango/Valores': '1=≤20 · 2=21–50 · 3=51–100 · 4=>100',                            'Uso': 'Predictor'},
    # --- Horas ---
    {'Variable': 'C318_T',   'Tipo': 'Numérica',   'Descripción': 'Total horas trabajadas en ocupación principal',   'Rango/Valores': '0–98 (99=missing)',                                               'Uso': 'Predictor'},
    {'Variable': 'C328_T',   'Tipo': 'Numérica',   'Descripción': 'Horas trabajadas en ocupaciones secundarias',     'Rango/Valores': '0–98 (99=missing)',                                               'Uso': 'Predictor'},
    {'Variable': 'whoraT',   'Tipo': 'Numérica',   'Descripción': 'Total horas trabajadas (todas las ocupaciones)',  'Rango/Valores': '0–998 (999=missing)',                                             'Uso': 'Predictor'},
    {'Variable': 'C331',     'Tipo': 'Numérica',   'Descripción': 'Horas normales semanales (todas las ocupaciones)','Rango/Valores': '0–998 (99=missing)',                                              'Uso': 'Predictor'},
    # --- Ingresos ---
    {'Variable': 'INGTOT',   'Tipo': 'Numérica',   'Descripción': 'Ingresos totales mensuales del trabajo',          'Rango/Valores': '0–999998 (999999=missing)',                                       'Uso': 'Predictor'},
    {'Variable': 'INGTOTP',  'Tipo': 'Numérica',   'Descripción': 'Ingreso principal mensual',                       'Rango/Valores': '0–999998 (999999=missing)',                                       'Uso': 'Predictor'},
    {'Variable': 'ingtrabw', 'Tipo': 'Numérica',   'Descripción': 'Ingreso mensual laboral (winsorizado)',            'Rango/Valores': '0–999998',                                                       'Uso': 'Predictor'},
    # --- Educación ---
    {'Variable': 'C366',     'Tipo': 'Ordinal',    'Descripción': 'Nivel educativo máximo aprobado',                 'Rango/Valores': '1=Sin nivel … 12=Maestría/Doctorado',                             'Uso': 'Predictor'},
    # --- Salud ---
    {'Variable': 'SEGURO1',  'Tipo': 'Categórica', 'Descripción': 'Indicador de afiliación a algún seguro de salud', 'Rango/Valores': 'Ver sub-variables C361_*',                                        'Uso': 'Predictor'},
    # --- Pensiones ---
    {'Variable': 'C364_1',   'Tipo': 'Binaria',    'Descripción': 'Afiliado al Sistema Privado de Pensiones (AFP)',  'Rango/Valores': '1=Sí · 2=No',                                                    'Uso': 'Predictor'},
    {'Variable': 'C364_2',   'Tipo': 'Binaria',    'Descripción': 'Afiliado al Sistema Nacional de Pensiones (SNP)','Rango/Valores': '1=Sí · 2=No',                                                    'Uso': 'Predictor'},
    # --- Etnicidad ---
    {'Variable': 'C376',     'Tipo': 'Categórica', 'Descripción': 'Lengua materna aprendida en la niñez',            'Rango/Valores': '1=Quechua … 10=Castellano · 12=Otra extranjera',                  'Uso': 'Predictor (ético)'},
    {'Variable': 'C377',     'Tipo': 'Categórica', 'Descripción': 'Autoidentificación étnica',                       'Rango/Valores': '1=Quechua … 7=Mestizo · 8=Otro',                                  'Uso': 'Predictor (ético)'},
    # --- Factor de expansión ---
    {'Variable': 'fa_son24', 'Tipo': 'Numérica',   'Descripción': 'Factor de expansión trimestral Set-Oct-Nov 2024', 'Rango/Valores': 'Continuo positivo',                                               'Uso': 'No usar en modelo base'},
])
diccionario

,Variable,Tipo,Descripción,Rango/Valores,Uso
0,REGION,Categórica,Región geográfica,1=Lima Met. · 2=Resto urbano · 3=Rural,Filtro/cobertura
1,ESTRATO,Categórica,Estrato geográfico del diseño muestral,1–8,Diseño muestral
2,RESIDENT,Categórica,Residente habitual del hogar,0=No · 1=Sí · 9=Omisión,Filtro
3,C207,Categórica,Sexo,1=Hombre · 2=Mujer,Predictor
4,C208,Numérica,Edad en años cumplidos,0–98 (99=missing),Predictor / filtro ≥14
5,C203,Categórica,Relación de parentesco con el jefe del hogar,1=Jefe … 11=Otro no pariente · 98=No residente,Predictor
6,OCUP300,Categórica,Nivel de ocupación / condición de actividad,1=Ocupado · 2=Desoc. abierto · 3=Desoc. oculto...,Filtro (solo ocupados)
7,P209H,Binaria,Voluntad Y disponibilidad para trabajar más horas,1=Sí · 2=No · vacío=No aplica,TARGET
8,C333,Categórica,¿Quería trabajar más horas?,1=Sí · 2=No,EXCLUIR (leakage)
9,C334,Categórica,¿Estuvo disponible para trabajar más horas?,1=Sí · 2=No,EXCLUIR (leakage)


## 2. Calidad de datos – variables relevantes del proyecto

In [5]:
VARS_RELEVANTES = [
    'REGION', 'ESTRATO', 'RESIDENT',
    'C207', 'C208', 'C203',
    'OCUP300', 'P209H', 'C333', 'C334',
    'C310', 'C311', 'C312', 'C313', 'C317',
    'C318_T', 'C328_T', 'whoraT', 'C331',
    'INGTOT', 'INGTOTP', 'ingtrabw',
    'C366', 'SEGURO1', 'C364_1', 'C364_2',
    'C376', 'C377',
]

# Filtrar solo columnas presentes en el snapshot
vars_presentes = [v for v in VARS_RELEVANTES if v in df.columns]
vars_ausentes  = [v for v in VARS_RELEVANTES if v not in df.columns]
if vars_ausentes:
    print(f'Advertencia – variables no encontradas en el snapshot: {vars_ausentes}')

calidad = pd.DataFrame({
    'Variable': vars_presentes,
    'Dtype':    [str(df[v].dtype) for v in vars_presentes],
    'Nulos':    [df[v].isnull().sum() for v in vars_presentes],
    'Nulos_%':  [(df[v].isnull().mean() * 100).round(2) for v in vars_presentes],
    'Únicos':   [df[v].nunique() for v in vars_presentes],
    'Ejemplo':  [df[v].dropna().iloc[0] if df[v].dropna().shape[0] > 0 else None for v in vars_presentes],
})
calidad

,Variable,Dtype,Nulos,Nulos_%,Únicos,Ejemplo
0,REGION,int64,0,0.0,1,1
1,ESTRATO,str,0,0.0,2,1
2,RESIDENT,int64,0,0.0,2,1
3,C207,int64,0,0.0,2,2
4,C208,str,0,0.0,103,60
5,C203,int64,0,0.0,10,1
6,OCUP300,str,0,0.0,6,1
7,P209H,str,0,0.0,3,2
8,C333,str,0,0.0,3,2
9,C334,str,0,0.0,3,


## 3. Notas y observaciones

- **`C208` (edad):** viene como `object` desde el Excel; requiere conversión a numérico antes de aplicar el filtro `>= 14`.
- **`P209H` (target):** presenta ~54% de valores nulos/vacíos. Corresponde a personas no ocupadas o a quienes la pregunta no aplica. Se excluirán aplicando el filtro `OCUP300 == 1`.
- **Variables de ingresos (`INGTOT`, `INGTOTP`, `ingtrabw`):** pueden contener el código `999999` como valor faltante; se reemplazará por `NaN` en preprocesamiento.
- **Variables binarias (Sí/No):** el código `9` representa missing value en varias de ellas (ej. `C364_*`, `C375_*`).
- **`C333` y `C334`:** se excluirán del conjunto de predictores por riesgo de **data leakage** al ser los componentes directos del target `P209H`.
- **Factor de expansión `fa_son24`:** no se usará como predictor; puede emplearse para análisis descriptivos ponderados.